In [1]:
import os
from dotenv import load_dotenv
from src.model_compare import compare_models
from src.chat import create_client, chat_turn
import pandas as pd

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
client = create_client(api_key)
print("Client ready.")

Client ready.


In [2]:
MODELS = [
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    #"openai/gpt-oss-120b:free",
    "poolside/laguna-m.1:free",
]

PROMPT = "Explain what a neural network is in 3 sentences, like I am 16 years old."

results = compare_models(client, MODELS, PROMPT)

for r in results:
    print(f"\n--- {r['model']} ({r['time']:.2f}s) ---")
    print(r['response'])

Querying nvidia/nemotron-3-ultra-550b-a55b:free...
Querying poolside/laguna-m.1:free...

--- nvidia/nemotron-3-ultra-550b-a55b:free (26.01s) ---
A neural network is a computer system modeled loosely on the human brain, built from layers of interconnected "neurons" that pass data to one another. Instead of being explicitly programmed with rules, it learns by analyzing massive amounts of examples—like photos or text—and automatically adjusts the strength of its internal connections to minimize errors. Once trained, it can recognize complex patterns in new data it has never seen before, powering things like facial recognition, language translation, and recommendation algorithms.

--- poolside/laguna-m.1:free (9.66s) ---

A neural network is a computer system designed like the human brain, with layers of "neurons" that process information. Each layer analyzes different features of the input (like edges in an image or words in a sentence) and passes the data forward, adjusting connections b

In [3]:
df = pd.DataFrame([
    {
        "Model": r["model"],
        "Response Time (s)": round(r["time"], 2),
        "Response Preview": r["response"][:120] + "..."
    }
    for r in results
])
df

,Model,Response Time (s),Response Preview
0,nvidia/nemotron-3-ultra-550b-a55b:free,26.01,A neural network is a computer system modeled ...
1,poolside/laguna-m.1:free,9.66,\nA neural network is a computer system design...


In [4]:
from src.constants import ASSISTANT_COLOR, USER_COLOR, BOLD, RESET, SYSTEM_COLOR, INFO_COLOR
# Import the colorizer function from your chat module
from src.chat import chat_turn, colorize_markdown 

MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
history = []

questions = [
    "What is a transformer in machine learning?",
    "How is it different from an RNN?",
    "Can you give me a real world example of where transformers are used?"
]

for question in questions:
    print(f"{USER_COLOR}{BOLD}You:{RESET} {question}")
    
    reply, history, error = chat_turn(client, MODEL, history, question)
    
    if error:
        print(f"Error: {error}\n")
    else:
        # Step 1: Format the raw markdown string
        formatted_reply = colorize_markdown(reply)
        
        # Step 2: Print the formatted version
        print(f"{ASSISTANT_COLOR}{BOLD}Assistant:{RESET}\n{ASSISTANT_COLOR}{formatted_reply}{RESET}\n")

You: What is a transformer in machine learning?
Error: 'NoneType' object is not subscriptable

You: How is it different from an RNN?
Assistant:
The fundamental difference lies in how they process sequence data: RNNs process sequentially (step-by-step), while Transformers process in parallel (all at once).

This single architectural shift cascades into massive differences in performance, training speed, and ability to handle long-range context.

Here is a detailed breakdown:

---

1. Core Architecture: Recurrence vs. Self-Attention

| Feature | RNN (LSTM / GRU) | Transformer |
| :--- | :--- | :--- |
| Mechanism | Recurrence: Maintains a hidden state ($h_t$) passed from one time step to the next. $h_t = f(h_{t-1}, x_t)$. | Self-Attention: Computes relationships between all tokens simultaneously using Query, Key, Value matrices. No hidden state passed between steps. |
| Data Flow | Sequential: Token 2 cannot be processed until Token 1 is finished. | Parallel: All tokens (Token 1, 2, 3... 

## Week 2 Findings

### Model Comparison
Three models were compared on the same prompt via OpenRouter API.
Response time and quality were recorded in the table above.
- mistral-7b is fastest and cheapest, good for high volume tasks
- llama-3-8b gives more detailed responses, good balance of quality and cost
- claude-3-haiku gives the most natural responses, best quality among the three

### Multi-Turn Chat
The chat function in src/chat.py maintains full conversation history
by appending each message to a list and sending the entire list on
every API call. This is how multi-turn conversation is simulated
since LLMs are stateless between calls. The third question above
references the previous answers proving memory works correctly.

### Local Models
llama3 and mistral were also run locally via Ollama. Local models
have zero API cost and full privacy but are slower and less capable
than hosted models like Claude.

### Key Concepts Learned
- Tokens: LLMs read chunks of text not individual words
- Temperature: controls randomness of output, 0 is deterministic, 1 is creative
- Context window: how much text the model can see at once
- System prompt: hidden instruction given to the model before conversation starts